In [1]:
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

In [2]:
X,y = make_classification(n_samples=10000, n_features=20, n_informative=3)

In [3]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train, y_train)
y_pred = dt.predict(x_test)

accuracy_score(y_test, y_pred)

0.8715

# Bagging

In [5]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),       ## base_estimator
    n_estimators=500,                     ### 500 Models
    max_samples=0.5,                  ### 50% of the training data
    bootstrap=True,                      ## With replacement
    random_state=42
)

In [6]:
bag.fit(x_train, y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.5,
                  n_estimators=500, random_state=42)

In [7]:
y_pred_bag = bag.predict(x_test)

In [8]:
accuracy_score(y_test, y_pred_bag)

0.914

In [9]:
bag.estimators_samples_[0].shape

(4000,)

In [10]:
bag.estimators_features_[0].shape

(20,)

# Bagging using SVM

In [ ]:
bg = BaggingClassifier(
    estimator=SVC(probability=True),  ## base_estimator
    n_estimators=500,                 ### 500 Models
    max_samples=0.25,                  ### 25% of the training data
    bootstrap=True,                    ## With replacement
    random_state=42
)

In [13]:
bg.fit(x_train, y_train)
y_pred_bg = bg.predict(x_test)
accuracy_score(y_test, y_pred_bg)

0.902

# Pasting

In [16]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,  ## Without replacement
    random_state=42,
    verbose=1,
    n_jobs=-1  ## Use all available cores

)

In [17]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:   12.5s remaining:   37.6s
[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:   12.8s finished
[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    0.3s remaining:    1.1s


Pasting classifier 0.912


[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    0.7s finished


# Random Subspaces

In [18]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,              ### Without row replacement
    max_features=0.5,
    bootstrap_features=True,                  ### With column replcement
    random_state=42


)

In [19]:
bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.9085


In [20]:

bag.estimators_samples_[0].shape

(8000,)

In [21]:

bag.estimators_features_[0].shape

(10,)

# Random Pathces

In [24]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)
     

In [25]:

bag.fit(x_train,y_train)
y_pred = bag.predict(x_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.902


# OBB Score

In [26]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)

In [27]:
bag.fit(x_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [28]:
bag.oob_score_

0.906875

In [29]:
y_pred = bag.predict(x_test)
print("Accuracy",accuracy_score(y_test,y_pred))
     

Accuracy 0.911


# Bagging Tips

### Bagging generally gives better results than Pasting
### Good results come around the 25% to 50% row sampling mark
### Random patches and subspaces should be used while dealing with high dimensional data
### To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

# Apply GridSearchCV

In [30]:
from sklearn.model_selection import GridSearchCV

In [31]:
parameters = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.1, 0.4, 0.7, 1.0],
    'bootstrap': [True, False],
    'max_features': [0.1, 0.4, 0.7, 1.0]
}

In [32]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)

In [ ]:
search.fit(x_train,y_train)

In [ ]:
search.best_score_

In [ ]:

search.best_params_